# v0.19.0 — Live queries (base)

A **live query** is a server-side subscription: SurrealDB pushes a notification every time a
record in the watched table is created, updated or deleted. This version ships the *raw* layer —
notifications arrive as the server's own envelope, not yet deserialized into model instances.

Live queries need a **WebSocket** connection and work on **both SurrealDB 2.6.x and 3.2.x**. The
two places the lines differ are normalised by the ORM, and this notebook shows both:

- SurrealDB 3.x refuses to watch a table that does not exist; 2.6.x accepts and stays silent.
- After `kill()`, 3.x sends a final `KILLED` notification and 2.6.x sends nothing — the ORM ends
  the stream itself, so your `async for` terminates the same way on either line.

Everything below runs top to bottom on either server.

## 1. Connect

In [1]:
import asyncio
import contextlib
import os

from surreal_orm_lite import LiveAction, SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

# Live queries require WebSocket: the SDK's HTTP connection raises NotImplementedError.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. A model, and an idempotent reset

In [2]:
from surreal_orm_lite import BaseSurrealModel


class Message(BaseSurrealModel):
    id: str
    text: str = ""


client = await SurrealDBConnectionManager.get_client()

# Re-running the notebook must not depend on leftovers from the previous run.
with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Message;", {})

# SurrealDB 3.x refuses a live query on an undefined table, so define it up front.
await client.query("DEFINE TABLE Message SCHEMALESS;", {})
print("Table Message ready")

Table Message ready


## 3. `watch()` — the everyday form

`watch()` is an async context manager: it starts the live query on entry and **kills it on exit**,
including when the body raises. That matters, because a live query left running holds a
server-side subscription until the connection drops.

The writes below happen in a separate task, so this cell shows what a real application sees:
changes made elsewhere arriving as they happen. Every demo is wrapped in a timeout, so a
notebook can never hang on a stream that stays silent.

In [3]:
async def emit_changes(delay: float = 0.4) -> None:
    """Make the three changes a live query reports, from another task."""
    await asyncio.sleep(delay)
    conn = await SurrealDBConnectionManager.get_client()
    await conn.query("CREATE Message:hello SET text = 'hello';", {})
    await conn.query("UPDATE Message:hello SET text = 'hello again';", {})
    await conn.query("DELETE Message:hello;", {})


async def watch_three() -> list[dict]:
    writer = asyncio.create_task(emit_changes())
    seen: list[dict] = []
    async with Message.objects().watch() as stream:
        print("live_id:", stream.live_id, "| active:", stream.is_active)
        async for notif in stream:
            seen.append(notif)
            print(f"  {notif['action']:<6} -> {notif['result']}")
            if len(seen) == 3:
                break
    await writer
    return seen


notifications = await asyncio.wait_for(watch_three(), timeout=30)
print("The subscription is killed on the way out of the block.")

live_id: 0ca3872d-02dd-46ec-84d4-a44c14ca9a2a | active: True


  CREATE -> {'id': RecordID(table_name=Message, record_id='hello'), 'text': 'hello'}
  UPDATE -> {'id': RecordID(table_name=Message, record_id='hello'), 'text': 'hello again'}
  DELETE -> {'id': RecordID(table_name=Message, record_id='hello'), 'text': 'hello again'}
The subscription is killed on the way out of the block.


## 4. What a notification looks like

Each notification is the server's **raw envelope**, a plain dict. Values keep their SDK types
(`RecordID`, `Datetime`) rather than being coerced — deserializing them into model instances is
v0.20.0.

| Key       | Meaning                                                      |
| --------- | ------------------------------------------------------------ |
| `action`  | `CREATE`, `UPDATE` or `DELETE`                               |
| `id`      | the live query's own uuid                                    |
| `record`  | the affected record                                          |
| `result`  | the record after the change; for `DELETE`, its last content  |
| `session` | SurrealDB 3.x only — absent on 2.6.x                         |

In [4]:
first = notifications[0]
for key, value in first.items():
    print(f"{key:<8} {value!r}")

print()
print("Actions in order:", [n["action"] for n in notifications])
print("'session' present (SurrealDB 3.x only):", "session" in first)

action   'CREATE'
id       UUID('0ca3872d-02dd-46ec-84d4-a44c14ca9a2a')
record   RecordID(table_name=Message, record_id='hello')
result   {'id': RecordID(table_name=Message, record_id='hello'), 'text': 'hello'}
session  UUID('bdba32e7-2b72-4cdb-b2e8-12ae5dfe1964')

Actions in order: ['CREATE', 'UPDATE', 'DELETE']
'session' present (SurrealDB 3.x only): True


### Comparing an action

`LiveAction` is a `StrEnum`, so it compares directly against the raw string in the envelope. You
never need to quote a magic value, and the envelope stays a plain dict.

In [5]:
deleted = [n for n in notifications if n["action"] == LiveAction.DELETE]
print("DELETE notifications:", len(deleted))
print("The raw value really is a plain string:", repr(notifications[0]["action"]))
print("…and it equals the enum member:", notifications[0]["action"] == LiveAction.CREATE)

DELETE notifications: 1
The raw value really is a plain string: 'CREATE'
…and it equals the enum member: True


## 5. The explicit form: `live()`, `subscribe_live()`, `kill()`

Use the three primitives directly when you need the uuid itself — to hand it to another task, or
to kill the subscription from somewhere else.

`subscribe_live()` is **not** a coroutine: call it without `await`. Buffering starts the moment
you call it rather than at the first iteration, which is why the write below is still reported
even though it happens before anything reads the stream.

In [6]:
live_id = await Message.objects().live()
print("live_id:", live_id)

stream = SurrealDBConnectionManager.subscribe_live(live_id)   # no await — buffering starts here

await client.query("CREATE Message:buffered SET text = 'written before the first read';", {})


async def read_one() -> dict:
    async for notif in stream:
        return notif
    raise AssertionError("stream ended without a notification")


notif = await asyncio.wait_for(read_one(), timeout=30)
print("Recovered the write made before the first read:", notif["result"]["text"])

# Returning from inside `async for` leaves the generator suspended; close it explicitly so the
# pattern a reader copies from here does not leak one.
await stream.aclose()
await SurrealDBConnectionManager.kill(live_id)
print("Killed.")

live_id: 61e6f0bb-bf74-4304-935d-2f98c3fe73d3
Recovered the write made before the first read: written before the first read
Killed.


### `kill()` ends the stream, on both server lines

This is the part that needs the ORM. SurrealDB 3.x sends a final `KILLED` notification when a
live query is killed; **2.6.x sends nothing at all**, which would leave a reader's `async for`
waiting forever. The ORM pushes its own end-of-stream marker, so the loop below terminates on
either line.

In [7]:
async def drain_until_killed() -> int:
    lq = await Message.objects().live()
    incoming = SurrealDBConnectionManager.subscribe_live(lq)

    async def consume() -> int:
        count = 0
        async for _ in incoming:      # ends only when the live query is killed
            count += 1
        return count

    consumer = asyncio.create_task(consume())
    await asyncio.sleep(0.3)
    await SurrealDBConnectionManager.kill(lq)
    return await consumer


received = await asyncio.wait_for(drain_until_killed(), timeout=30)
print("The `async for` ended by itself after kill(); notifications seen:", received)

The `async for` ended by itself after kill(); notifications seen: 0


In [8]:
# kill() is idempotent: an unknown or already-killed uuid is a no-op, like the ORM's other
# cleanup-on-a-missing-target operations.
from uuid import uuid4

await SurrealDBConnectionManager.kill(uuid4())
print("kill() on an unknown uuid: no error")

kill() on an unknown uuid: no error


## 6. What a live query refuses

v0.19.0 watches a **whole table**. A queryset carrying a clause that could not be honoured is
rejected by name rather than silently watched whole — the filtered form (`LIVE SELECT … WHERE`)
lands in v0.20.0.

In [9]:
from surreal_orm_lite.exceptions import SurrealDbError

try:
    await Message.objects().filter(text="hello").limit(5).live()
except SurrealDbError as exc:
    print("Refused:", exc)

Refused: Live queries in v0.19.0 watch a whole table and cannot honour filter(), limit(). Drop the clause, or wait for the filtered form (`LIVE SELECT ... WHERE`) landing in v0.20.0.


Live queries also cannot run over HTTP. The SDK would raise a bare `NotImplementedError`; the ORM
checks the configured transport first and explains the requirement.

In [10]:
SurrealDBConnectionManager.set_connection(
    url=f"http://{HOST}:{PORT}",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
try:
    await Message.objects().live()
except SurrealDbError as exc:
    print("Refused:", exc)
finally:
    await SurrealDBConnectionManager.close_connection()
    SurrealDBConnectionManager.set_connection(
        url=f"ws://{HOST}:{PORT}/rpc",
        user="root",
        password="root",
        namespace="examples",
        database="examples",
    )
    client = await SurrealDBConnectionManager.get_client()
print("Back on WebSocket.")

Refused: Live queries require a WebSocket connection (ws:// or wss://); the configured URL is 'http://localhost:8001'. Call SurrealDBConnectionManager.set_connection() with a ws:// URL.
Back on WebSocket.


## 7. The one behaviour that differs by server line

SurrealDB 3.x refuses to watch a table that does not exist. SurrealDB 2.6.x accepts the
subscription and simply never notifies. The ORM reports the difference rather than papering over
it — subscribing is not a reason to create a table — so it normalises the 3.x error into
`SurrealDbNotFoundError` with a message telling you what to do.

The probe below branches the way the test suite does, so this cell runs cleanly on either line.

In [11]:
from surreal_orm_lite.exceptions import SurrealDbNotFoundError


class Ghost(BaseSurrealModel):
    id: str


with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Ghost;", {})

try:
    ghost_id = await Ghost.objects().live()
except SurrealDbNotFoundError as exc:
    print("SurrealDB 3.x — strict:")
    print(" ", exc)
else:
    print("SurrealDB 2.6.x — lenient: the subscription was accepted and will stay silent.")
    print("  live_id:", ghost_id)
    await SurrealDBConnectionManager.kill(ghost_id)

SurrealDB 3.x — strict:
  Cannot start a live query on 'Ghost': the table does not exist. SurrealDB 3.x refuses to watch an undefined table (2.6.x accepts it and stays silent). Create it first, e.g. DEFINE TABLE Ghost SCHEMALESS.


## 8. Cleanup

Kill nothing by hand here: every live query above was either killed explicitly or by the
`watch()` block. Only the demo table is left to remove.

In [12]:
with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Message;", {})
await SurrealDBConnectionManager.close_connection()
print("Cleaned up.")

Cleaned up.


## What is next

- **v0.20.0** — a typed `LiveQuerySet`: notifications deserialized into model instances, filtered
  live queries through `LIVE SELECT … WHERE`, and `DIFF` mode. The installed SDK accepts
  `live(diff=True)` but never puts the flag on the wire, so v0.19.0 deliberately does not expose
  it.
- **v0.21.0** — automatic reconnect and resubscribe.

### What ends a stream today

`kill()` ends it, leaving a `watch()` block ends it, and so does `close_connection()` — the ORM
releases every reader on the loop before the socket goes, so an ordinary shutdown cannot leave a
wedged task behind. A WebSocket that drops on its **own** does not: the SDK's receive task
absorbs the close without telling live-query subscribers, so the iterator stays suspended until
you call `kill()` or close the connection.